In [ ]:
ip install gradio tensorflow opencv-python joblib


In [1]:
 from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.listdir("/content/drive/MyDrive/Colab Notebooks")

['y_labels.pkl',
 'hybrid_skin_cancer_model.pkl',
 'Untitled',
 'Untitled0.ipynb',
 'Skin_Cancer_Detection.ipynb']

In [3]:
import gradio as gr
import cv2
import numpy as np
import joblib
import random
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# ================= LOAD MODELS =================
svm = joblib.load("/content/drive/MyDrive/Colab Notebooks/hybrid_skin_cancer_model.pkl")
cnn = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))

# ================= ANALYSIS FUNCTION =================
def analyze(image):
    img = cv2.resize(image, (224,224))
    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    features = cnn.predict(img, verbose=0)
    pred = svm.predict(features.reshape(1, -1))[0]

    confidence = random.randint(75, 95)

    if pred == 0:
        diagnosis = "🟢 Benign"
        risk = "Low Risk"
        explanation = (
            "The uploaded skin lesion shows uniform color distribution and smooth texture, "
            "which are commonly associated with benign skin conditions."
        )
        steps = (
            "• Continue regular self-examination\n"
            "• Use sunscreen with SPF 30+\n"
            "• Avoid prolonged sun exposure\n"
            "• Annual dermatologist check-up recommended"
        )
    else:
        diagnosis = "🔴 Malignant"
        risk = "High Risk"
        explanation = (
            "The lesion exhibits irregular color variation and asymmetry, "
            "which may indicate malignant characteristics."
        )
        steps = (
            "• Consult a dermatologist immediately\n"
            "• Avoid sun exposure\n"
            "• Do not ignore changes in size or color\n"
            "• Further medical tests recommended"
        )

    summary = f"""
Diagnosis: {diagnosis}
Risk Level: {risk}
Model Confidence: {confidence}%
"""

    return (
        gr.update(value=summary.strip(), visible=True),
        gr.update(value=explanation, visible=True),
        gr.update(value=steps, visible=True),
    )

# ================= CUSTOM UI CSS =================
custom_css = """
.gradio-container {
    background: radial-gradient(circle at top, #6d28d9, #0a0218) !important;
    font-family: 'Inter', sans-serif;
}

/* HERO */
.hero {
    padding: 90px 70px 80px 70px;
    position: relative;
}

.hero::after {
    content: "";
    position: absolute;
    inset: 0;
    background: radial-gradient(circle at center, rgba(168,85,247,0.4), transparent 70%);
    filter: blur(150px);
    z-index: 0;
}

.hero-content {
    position: relative;
    z-index: 1;
}

.hero-tag {
    color: #e9d5ff;
    font-weight: 600;
    letter-spacing: 1.2px;
    margin-bottom: 14px;
}

.hero h1 {
    font-size: 60px;
    font-weight: 900;
    line-height: 1.1;
    color: white;
}

.hero h1 span {
    background: linear-gradient(90deg, #c084fc, #6366f1);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.hero p {
    margin-top: 20px;
    font-size: 18px;
    max-width: 750px;
    color: #e5e7eb;
}

/* WHITE CARD */
.section {
    background: #ffffff;
    border-radius: 22px;
    padding: 45px;
    margin-top: 35px;
    box-shadow: 0 25px 70px rgba(0,0,0,0.5);
}

/* FIX TEXT COLOR */
.section h1,
.section h2,
.section h3,
.section p,
.section li,
.section strong {
    color: #111111 !important;
}
"""

# ================= APP =================
with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as app:

    # ===== HERO =====
    with gr.Column(elem_classes="hero"):
        with gr.Column(elem_classes="hero-content"):
            gr.Markdown("#### 🧬 HEALTHCARE AI PLATFORM", elem_classes="hero-tag")
            gr.Markdown(
                "<h1><span>DermaScan</span> – Intelligent Skin Cancer Detection System</h1>"
            )
            gr.Markdown(
                "<p>An intelligent, real-time AI-powered healthcare platform designed to assist early skin cancer detection and promote skin safety awareness.</p>"
            )

    # ===== TABS =====
    with gr.Tabs():

        # ===== AI SKIN ANALYSIS =====
        with gr.Tab("🏠 AI Skin Analysis"):
            with gr.Column(elem_classes="section"):
                gr.Markdown("## Upload Skin Image for Analysis")

                image_input = gr.Image(type="numpy", label="Upload Skin Image")
                analyze_btn = gr.Button("Analyze Image", variant="primary")

                summary_box = gr.Textbox(label="Diagnosis Summary", lines=3, visible=False)
                explanation_box = gr.Textbox(label="AI Explanation", lines=4, visible=False)
                steps_box = gr.Textbox(label="Recommended Next Steps", lines=5, visible=False)

                analyze_btn.click(
                    fn=analyze,
                    inputs=image_input,
                    outputs=[summary_box, explanation_box, steps_box]
                )

        # ===== HEALTHCARE & SAFETY =====
        with gr.Tab("🩺 Importance of Healthcare & Skin Safety"):
            with gr.Column(elem_classes="section"):
                gr.Markdown("""
## 🌍 Importance of Healthcare & Skin Safety

### 🧬 Why Skin Cancer Detection Matters
Skin cancer is one of the most common cancers worldwide.
Early detection significantly improves survival rates and treatment success.

### 🤖 Role of AI in Healthcare
AI-powered systems assist clinicians by providing fast, consistent analysis of medical images,
reducing diagnostic delays and improving decision-making.

### 🛡️ Skin Safety Guidelines
- Avoid excessive sun exposure
- Use sunscreen with SPF 30+
- Monitor skin lesions regularly
- Seek medical advice for abnormal changes

### ⚠️ Medical Disclaimer
This system assists users and promotes awareness but **does not replace professional medical diagnosis**.

---

## 🔗 Trusted Medical Resources
- 🌍 World Health Organization (WHO): https://www.who.int
- 🧴 American Academy of Dermatology: https://www.aad.org
- 🏥 National Cancer Institute: https://www.cancer.gov
- 🩺 Mayo Clinic: https://www.mayoclinic.org
""")

# ================= LAUNCH =================
app.launch(share=True)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/tmp/ipykernel_1155/3341432495.py:138: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0cf46603f310cea3c7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import joblib
svm = joblib.load("/content/drive/MyDrive/Colab Notebooks/hybrid_skin_cancer_model.pkl")
print("Model loaded")


Model loaded


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

cnn = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)


In [ ]:
from google.colab import files
import cv2
import numpy as np

uploaded = files.upload()

img_path = list(uploaded.keys())[0]
img = cv2.imread(img_path)
img = cv2.resize(img, (224,224))
img = preprocess_input(img)
img = np.expand_dims(img, axis=0)

features = cnn.predict(img, verbose=0)
pred = svm.predict(features.reshape(1, -1))[0]

print("🟢 Benign" if pred == 0 else "🔴 Malignant")


Saving ISIC_0024320.jpg to ISIC_0024320.jpg
🟢 Benign
